# 07 Pilot LSTM Sequence Model

**Phase 4 - Sequential Learning Analytics**  
Research contract: `PHASE4_RESEARCH_CONTRACT_v1.md`  
Schema version: `lstm_v1`  
Backend: PyTorch (TensorFlow not available on Python 3.14)  
Prerequisites: M2 + M3 must be complete.

## CRITICAL PILOT LIMITATION

| Item | Value |
|---|---|
| `label_source` | `proxy_behavioral` |
| `label_validity` | `pilot_only` |
| Total learners | 10 (8 train, 2 test) |
| Minimum for thesis | >=60 learners, teacher-reviewed labels |

All metrics are **technical validation results only**.  
Do not interpret as proof of LSTM superiority or as final Chapter 4 results.

## Experiments

| ID | Model | Input |
|---|---|---|
| EXP-A | LSTM Sequence Only | `sequence_tensors_v1.npz` |
| EXP-B | LSTM Sequence + TAG Features | + `tag_graph_features_v1.parquet` (optional ablation) |


In [1]:
import os, json, hashlib, time, warnings, random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn

warnings.filterwarnings('ignore', category=FutureWarning)

SEQ_DIR   = Path('data/sequences')
TAG_DIR   = Path('data/tag')
MODEL_DIR = Path('models/sequence/lstm')
REP_DIR   = Path('../reports/phase4/lstm')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REP_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_VERSION       = 'lstm_v1'
M2_SCHEMA_VERSION    = 'seq_v1'
PHASE3_SOURCE_SHA    = '193b18949e40e6bd3bbfb70034a5772ce51d1b7e'
SEEDS                = [11, 22, 33, 42, 55]
VAL_SPLIT_SIZE       = 0.25
MAX_EPOCHS           = 200
EARLY_STOP_PATIENCE  = 15
LSTM_UNITS           = 32
DROPOUT_RATE         = 0.2
LEARNING_RATE        = 1e-3
BATCH_SIZE           = 16
DEVICE               = torch.device('cpu')  # CPU-compatible configuration

BLACKLISTED_X_FIELDS = {
    'at_risk', 'total_2c3l_score', 'grade_letter', 'is_teacher_reviewed',
    'c1_correctness_result_score', 'c2_semantic_consistency_score',
    'l1_logical_reasoning_score', 'l2_learning_process_score',
    'l3_difficulty_complexity_score', 'label_source', 'label_validity',
}

print(f'Schema version : {SCHEMA_VERSION}')
print(f'PyTorch version: {torch.__version__}')
print(f'Device         : {DEVICE}')
print(f'Seeds          : {SEEDS}')
print(f'Architecture   : Masking(pack) -> LSTM({LSTM_UNITS}) -> Dropout({DROPOUT_RATE}) -> Linear(sigmoid)')


Schema version : lstm_v1
PyTorch version: 2.13.0+cpu
Device         : cpu
Seeds          : [11, 22, 33, 42, 55]
Architecture   : Masking(pack) -> LSTM(32) -> Dropout(0.2) -> Linear(sigmoid)


In [2]:
manifest_path     = SEQ_DIR / 'sequence_manifest_v1.json'
tensors_path      = SEQ_DIR / 'sequence_tensors_v1.npz'
split_ledger_path = SEQ_DIR / 'split_assignments.parquet'
seq_index_path    = SEQ_DIR / 'sequence_index.parquet'

for label, p in [('manifest', manifest_path), ('tensors', tensors_path),
                  ('split_ledger', split_ledger_path), ('seq_index', seq_index_path)]:
    if not p.exists():
        raise FileNotFoundError(f'M2 artifact missing: {p}')
    print(f'  {label:16s}: {p}')

manifest     = json.loads(manifest_path.read_text())
split_ledger = pd.read_parquet(split_ledger_path)
seq_index    = pd.read_parquet(seq_index_path)

if manifest.get('schema_version') != M2_SCHEMA_VERSION:
    raise ValueError(f"M2 schema mismatch: {manifest.get('schema_version')}")

print(f"M2 manifest schema_version: {manifest['schema_version']}")
print(f"Input checksums: {list(manifest.get('input_files',{}).keys())}")


  manifest        : data\sequences\sequence_manifest_v1.json
  tensors         : data\sequences\sequence_tensors_v1.npz
  split_ledger    : data\sequences\split_assignments.parquet
  seq_index       : data\sequences\sequence_index.parquet
M2 manifest schema_version: seq_v1
Input checksums: ['sequence_csv', 'sequence_sha', 'attempt_csv', 'attempt_sha', 'outcome_csv', 'outcome_sha']


In [3]:
tensors = np.load(tensors_path)
X_train_full    = tensors['X_train'].astype(np.float32)
y_train_full    = tensors['y_train'].astype(np.int64)
mask_train_full = tensors['mask_train']
X_test          = tensors['X_test'].astype(np.float32)
y_test          = tensors['y_test'].astype(np.int64)
mask_test       = tensors['mask_test']

MAX_LEN    = X_train_full.shape[1]
N_FEATURES = X_train_full.shape[2]

print(f'X_train_full : {X_train_full.shape}  y_train_full : {y_train_full.shape}')
print(f'X_test       : {X_test.shape}         y_test       : {y_test.shape}')
print(f'MAX_LEN={MAX_LEN}  N_FEATURES={N_FEATURES}')
print(f'Train class dist : {dict(zip(*np.unique(y_train_full, return_counts=True)))}')
print(f'Test  class dist : {dict(zip(*np.unique(y_test, return_counts=True)))}')

split_map      = dict(zip(split_ledger['academy_member_id'], split_ledger['split']))
train_learners = sorted(split_ledger[split_ledger['split'] == 'train']['academy_member_id'].tolist())
test_learners  = sorted(split_ledger[split_ledger['split'] == 'test']['academy_member_id'].tolist())

train_seq_ids = [
    f"{row.academy_member_id}::{row.task_code}"
    for row in seq_index.itertuples()
    if split_map.get(row.academy_member_id) == 'train'
]
test_seq_ids = [
    f"{row.academy_member_id}::{row.task_code}"
    for row in seq_index.itertuples()
    if split_map.get(row.academy_member_id) == 'test'
]
train_learner_groups = np.array([
    row.academy_member_id
    for row in seq_index.itertuples()
    if split_map.get(row.academy_member_id) == 'train'
])

assert len(train_seq_ids) == X_train_full.shape[0]
assert len(test_seq_ids)  == X_test.shape[0]
print(f'\nTrain seqs: {len(train_seq_ids)}  Test seqs: {len(test_seq_ids)}')
print('Sequence ID alignment: OK')


X_train_full : (72, 6, 10)  y_train_full : (72,)
X_test       : (18, 6, 10)         y_test       : (18,)
MAX_LEN=6  N_FEATURES=10
Train class dist : {np.int64(0): np.int64(45), np.int64(1): np.int64(27)}
Test  class dist : {np.int64(0): np.int64(9), np.int64(1): np.int64(9)}

Train seqs: 72  Test seqs: 18
Sequence ID alignment: OK


In [4]:
tag_feat_path      = TAG_DIR / 'tag_graph_features_v1.parquet'
tag_manifest_path  = TAG_DIR / 'tag_manifest_v1.json'
TAG_AVAILABLE = tag_feat_path.exists() and tag_manifest_path.exists()

if TAG_AVAILABLE:
    tag_feat_df       = pd.read_parquet(tag_feat_path)
    tag_manifest      = json.loads(tag_manifest_path.read_text())
    TAG_FEATURE_NAMES = tag_manifest['graph_feature_names']
    leaked = sorted(set(tag_feat_df.columns) & BLACKLISTED_X_FIELDS)
    if leaked:
        raise ValueError(f'TAG leakage: {leaked}')
    n_tag         = len(TAG_FEATURE_NAMES)
    tag_by_sid    = dict(zip(tag_feat_df['sequence_id'], tag_feat_df[TAG_FEATURE_NAMES].values.tolist()))
    tag_train_full = np.array([tag_by_sid.get(s,[0.0]*n_tag) for s in train_seq_ids], dtype=np.float32)
    tag_test       = np.array([tag_by_sid.get(s,[0.0]*n_tag) for s in test_seq_ids],  dtype=np.float32)
    print(f'TAG features: {n_tag}  tag_train={tag_train_full.shape}  tag_test={tag_test.shape}')
else:
    TAG_FEATURE_NAMES=[]; tag_train_full=None; tag_test=None; n_tag=0
    print('TAG features not available -- EXP-B skipped')


TAG features: 18  tag_train=(72, 18)  tag_test=(18, 18)

In [5]:
pre_checks = []
def pre_chk(name, passed, detail=''):
    pre_checks.append({'check':name,'result':'PASS' if passed else 'FAIL','detail':str(detail)})
    print(f'  {"OK" if passed else "FAIL"} {name}' + (f' -- {detail}' if detail else ''))

print('-- Pre-training checks --')
pre_chk('Input checksums in M2 manifest',
    all(k in manifest.get('input_files',{}) for k in ['sequence_sha','attempt_sha','outcome_sha']),
    f"{len(manifest.get('input_files',{}))} entries")
pre_chk('Tensor shape valid (3-D)',
    X_train_full.ndim==3 and X_test.ndim==3,
    f'train={X_train_full.shape} test={X_test.shape}')
pre_chk('Padding mask shape matches tensor',
    mask_train_full.shape==X_train_full.shape[:2] and mask_test.shape==X_test.shape[:2],
    f'train_mask={mask_train_full.shape}')
no_nan = not(np.isnan(X_train_full).any() or np.isinf(X_train_full).any()
             or np.isnan(X_test).any() or np.isinf(X_test).any())
pre_chk('No NaN/Inf in tensors', no_nan)
ms = manifest.get('dataset_stats',{})
pre_chk('Frozen split unchanged',
    X_train_full.shape[0]==ms.get('train_shape',[0])[0] and
    X_test.shape[0]==ms.get('test_shape',[0])[0],
    f'train={X_train_full.shape[0]} test={X_test.shape[0]}')
overlap = set(train_learners) & set(test_learners)
pre_chk('No learner overlap', len(overlap)==0, f'overlap={len(overlap)}')
scaler_info = json.loads((SEQ_DIR/'scaler_v1.json').read_text()) if (SEQ_DIR/'scaler_v1.json').exists() else {}
pre_chk('Scaler fitted on training data only',
    scaler_info.get('fit_split')=='train',
    f"fit_split={scaler_info.get('fit_split','unknown')}")
blacklisted_in_x = sorted(set(manifest.get('parameters',{}).get('feature_names',[])) & BLACKLISTED_X_FIELDS)
pre_chk('No blacklisted fields in X', len(blacklisted_in_x)==0,
    f'blacklisted: {blacklisted_in_x}' if blacklisted_in_x else 'clean')

n_pre_fail = sum(1 for c in pre_checks if c['result']=='FAIL')
print(f'\n{len(pre_checks)-n_pre_fail}/{len(pre_checks)} pre-training checks passed')
if n_pre_fail > 0:
    raise RuntimeError(f'Pre-training validation FAILED ({n_pre_fail} checks).')


-- Pre-training checks --
  OK Input checksums in M2 manifest -- 6 entries
  OK Tensor shape valid (3-D) -- train=(72, 6, 10) test=(18, 6, 10)
  OK Padding mask shape matches tensor -- train_mask=(72, 6)
  OK No NaN/Inf in tensors
  OK Frozen split unchanged -- train=72 test=18
  OK No learner overlap -- overlap=0
  OK Scaler fitted on training data only -- fit_split=train
  OK No blacklisted fields in X -- clean

8/8 pre-training checks passed


In [6]:
class LSTMSeqOnly(nn.Module):
    """Masking -> LSTM -> Dropout -> Linear(sigmoid). Equivalent to Keras architecture."""
    def __init__(self, n_features, lstm_units, dropout_rate):
        super().__init__()
        self.lstm    = nn.LSTM(n_features, lstm_units, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc      = nn.Linear(lstm_units, 1)

    def forward(self, x, lengths):
        # Pack padded sequences (equivalent to Masking layer)
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)  # h_n: (1, B, H)
        h = h_n.squeeze(0)               # (B, H)
        h = self.dropout(h)
        return torch.sigmoid(self.fc(h)).squeeze(1)  # (B,)


class LSTMSeqTag(nn.Module):
    """Masking -> LSTM -> Dropout + TAG concat -> Linear(sigmoid)."""
    def __init__(self, n_features, n_tag, lstm_units, dropout_rate):
        super().__init__()
        self.lstm    = nn.LSTM(n_features, lstm_units, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc      = nn.Linear(lstm_units + n_tag, 1)

    def forward(self, x, lengths, tag):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h = h_n.squeeze(0)
        h = self.dropout(h)
        combined = torch.cat([h, tag], dim=1)
        return torch.sigmoid(self.fc(combined)).squeeze(1)


def set_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seeds(42)
demo = LSTMSeqOnly(N_FEATURES, LSTM_UNITS, DROPOUT_RATE)
n_params = sum(p.numel() for p in demo.parameters())
print(f'LSTMSeqOnly: {n_params} trainable parameters')
print(f'Architecture: Masking(pack_padded) -> LSTM({LSTM_UNITS}) -> Dropout({DROPOUT_RATE}) -> Linear(1, sigmoid)')


LSTMSeqOnly: 5665 trainable parameters
Architecture: Masking(pack_padded) -> LSTM(32) -> Dropout(0.2) -> Linear(1, sigmoid)


In [7]:
def make_val_split(X, y, groups, seed, val_size):
    gss = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    tr_idx, vl_idx = next(gss.split(X, y, groups))
    assert not (set(groups[tr_idx]) & set(groups[vl_idx])), 'SPLIT LEAKAGE'
    return tr_idx, vl_idx

tr_idx, vl_idx = make_val_split(
    X_train_full, y_train_full, train_learner_groups, seed=42, val_size=VAL_SPLIT_SIZE)
print(f'Train pool   : {len(train_learner_groups)} seqs ({len(set(train_learner_groups))} learners)')
print(f'Sub-train    : {len(tr_idx)} seqs ({len(set(train_learner_groups[tr_idx]))} learners)')
print(f'Validation   : {len(vl_idx)} seqs ({len(set(train_learner_groups[vl_idx]))} learners)')
print(f'Test (frozen): {X_test.shape[0]} seqs ({len(test_learners)} learners)')


Train pool   : 72 seqs (8 learners)
Sub-train    : 54 seqs (6 learners)
Validation   : 18 seqs (2 learners)
Test (frozen): 18 seqs (2 learners)


In [8]:
def compute_lengths(X_np, mask_np):
    """Compute true sequence lengths from mask (True = real step)."""
    lengths = mask_np.sum(axis=1).astype(np.int64)  # (B,)
    lengths = np.maximum(lengths, 1)  # at least 1 to avoid LSTM errors
    return lengths


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc  = float(accuracy_score(y_true, y_pred))
    ncls = len(np.unique(y_true))
    if ncls < 2:
        prec=rec=f1=auc=None
    else:
        prec = float(precision_score(y_true, y_pred, zero_division=0))
        rec  = float(recall_score(y_true, y_pred, zero_division=0))
        f1   = float(f1_score(y_true, y_pred, zero_division=0))
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = None
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {'accuracy':acc,'precision':prec,'recall':rec,'f1':f1,
            'roc_auc':auc,'confusion_matrix':cm,'n_classes_in_test':int(ncls)}


def run_epoch(model, X_t, y_t, mask_t, optimizer, criterion, cw_tensor,
              batch_size, tag_t=None, training=True):
    model.train(training)
    n = X_t.shape[0]
    indices = list(range(n))
    if training: random.shuffle(indices)
    total_loss = 0.0; n_batches = 0
    with torch.set_grad_enabled(training):
        for start in range(0, n, batch_size):
            idx   = indices[start:start+batch_size]
            xb    = X_t[idx]; yb = y_t[idx].float(); mb = mask_t[idx]
            lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
            if tag_t is not None:
                preds = model(xb, lens, tag_t[idx])
            else:
                preds = model(xb, lens)
            weights = torch.where(yb==1, cw_tensor[1], cw_tensor[0])
            loss = (criterion(preds, yb) * weights).mean()
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item(); n_batches += 1
    return total_loss / max(n_batches, 1)


def run_seed(seed, experiment, X_pool, y_pool, mask_pool, groups,
             X_te, y_te, mask_te, tag_pool=None, tag_te=None):
    set_seeds(seed)
    tr_idx, vl_idx = make_val_split(X_pool, y_pool, groups, seed, VAL_SPLIT_SIZE)
    X_tr = torch.tensor(X_pool[tr_idx]); y_tr = torch.tensor(y_pool[tr_idx])
    X_vl = torch.tensor(X_pool[vl_idx]); y_vl = torch.tensor(y_pool[vl_idx])
    mask_tr = torch.tensor(mask_pool[tr_idx]); mask_vl = torch.tensor(mask_pool[vl_idx])
    X_te_t  = torch.tensor(X_te); mask_te_t = torch.tensor(mask_te)

    # Class weights from sub-train only
    classes = np.unique(y_pool[tr_idx])
    if len(classes) == 2:
        cw_vals = compute_class_weight('balanced', classes=classes, y=y_pool[tr_idx])
        cw = torch.tensor([cw_vals[0], cw_vals[1]], dtype=torch.float32)
    else:
        cw = torch.ones(2, dtype=torch.float32)

    tag_tr_t = tag_vl_t = tag_te_t = None
    if experiment == 'seq_tag' and tag_pool is not None:
        tag_tr_t = torch.tensor(tag_pool[tr_idx])
        tag_vl_t = torch.tensor(tag_pool[vl_idx])
        tag_te_t = torch.tensor(tag_te)
        model = LSTMSeqTag(N_FEATURES, n_tag, LSTM_UNITS, DROPOUT_RATE).to(DEVICE)
    else:
        model = LSTMSeqOnly(N_FEATURES, LSTM_UNITS, DROPOUT_RATE).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss(reduction='none')

    history = {'loss':[], 'val_loss':[], 'accuracy':[], 'val_accuracy':[]}
    best_val_loss = float('inf'); patience_count = 0; best_state = None

    t0 = time.perf_counter()
    for epoch in range(MAX_EPOCHS):
        tr_loss = run_epoch(model, X_tr, y_tr, mask_tr, optimizer, criterion, cw,
                            BATCH_SIZE, tag_tr_t, training=True)
        vl_loss = run_epoch(model, X_vl, y_vl, mask_vl, optimizer, criterion, cw,
                            BATCH_SIZE, tag_vl_t, training=False)
        # Compute accuracy
        with torch.no_grad():
            lens_tr = mask_tr.sum(dim=1).clamp(min=1)
            lens_vl = mask_vl.sum(dim=1).clamp(min=1)
            p_tr = (model(X_tr, lens_tr, tag_tr_t) if tag_tr_t is not None else model(X_tr, lens_tr)).numpy()
            p_vl = (model(X_vl, lens_vl, tag_vl_t) if tag_vl_t is not None else model(X_vl, lens_vl)).numpy()
        tr_acc = float(accuracy_score(y_tr.numpy(), (p_tr>=0.5).astype(int)))
        vl_acc = float(accuracy_score(y_vl.numpy(), (p_vl>=0.5).astype(int)))
        history['loss'].append(tr_loss); history['val_loss'].append(vl_loss)
        history['accuracy'].append(tr_acc); history['val_accuracy'].append(vl_acc)
        # Early stopping (monitors val_loss, NOT test)
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss; patience_count = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= EARLY_STOP_PATIENCE:
                break
    train_t = time.perf_counter() - t0

    # Restore best weights
    if best_state:
        model.load_state_dict(best_state)

    # Inference on test
    model.eval()
    t1 = time.perf_counter()
    with torch.no_grad():
        lens_te = mask_te_t.sum(dim=1).clamp(min=1)
        y_prob = (model(X_te_t, lens_te, tag_te_t) if tag_te_t is not None
                  else model(X_te_t, lens_te)).numpy()
    inf_t = (time.perf_counter() - t1) / max(len(y_te), 1)

    m = compute_metrics(y_te, y_prob)
    m.update({'train_time_sec': round(train_t,4),
              'inf_time_per_seq_sec': round(inf_t,6),
              'epochs_trained': len(history['loss']),
              'seed': seed, 'experiment': experiment,
              'n_sub_train_seqs': int(len(tr_idx)),
              'n_val_seqs': int(len(vl_idx)),
              'n_test_seqs': int(len(y_te)),
              'class_weight': cw.tolist(),
              'label_validity': 'pilot_only', 'label_source': 'proxy_behavioral',
              'roc_auc_note': ('computed' if m['n_classes_in_test']>=2
                               else 'ROC-AUC undefined -- test set contains only one class')})
    return {'seed':seed,'experiment':experiment,'metrics':m,'history':history,
            'y_pred_prob':y_prob.tolist(),'y_true':y_te.tolist(),'model':model}


print('EXP-A: LSTM Sequence Only')
print(f'Running {len(SEEDS)} seeds: {SEEDS}\n')
exp_a_results = []
for seed in SEEDS:
    print(f'  Seed {seed:3d} ...', end=' ', flush=True)
    r = run_seed(seed, 'seq_only',
                 X_train_full, y_train_full, mask_train_full, train_learner_groups,
                 X_test, y_test, mask_test)
    m = r['metrics']
    auc = f"{m['roc_auc']:.4f}" if m['roc_auc'] is not None else 'NA'
    f1v = f"{m['f1']:.4f}"      if m['f1']      is not None else 'NA'
    print(f"acc={m['accuracy']:.4f}  f1={f1v}  auc={auc}  epochs={m['epochs_trained']}  "
          f"train={m['train_time_sec']:.1f}s")
    exp_a_results.append(r)
print('\nEXP-A complete.')


EXP-A: LSTM Sequence Only
Running 5 seeds: [11, 22, 33, 42, 55]

  Seed  11 ... 

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

...

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

acc=1.0000  f1=1.0000  auc=1.0000  epochs=200  train=3.0s

EXP-A complete.


C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

In [9]:
def summarise_seeds(results, label):
    keys = ['accuracy','precision','recall','f1','roc_auc',
            'train_time_sec','inf_time_per_seq_sec','epochs_trained']
    s = {'experiment':label, 'seeds':[r['seed'] for r in results]}
    for k in keys:
        vals = [r['metrics'][k] for r in results if r['metrics'].get(k) is not None]
        s[f'{k}_mean'] = round(float(np.mean(vals)),6) if vals else None
        s[f'{k}_std']  = round(float(np.std(vals)), 6) if vals else None
    s['per_seed'] = [r['metrics'] for r in results]
    return s

exp_a_summary = summarise_seeds(exp_a_results, 'EXP-A: LSTM Sequence Only')
print('EXP-A Summary')
print(f"  accuracy : {exp_a_summary['accuracy_mean']:.4f} +/- {exp_a_summary['accuracy_std']:.4f}")
print(f"  f1       : {exp_a_summary['f1_mean']} +/- {exp_a_summary['f1_std']}")
print(f"  roc_auc  : {exp_a_summary['roc_auc_mean']} +/- {exp_a_summary['roc_auc_std']}")
print(f"  epochs   : {exp_a_summary['epochs_trained_mean']:.1f} +/- {exp_a_summary['epochs_trained_std']:.1f}")
print('WARNING: pilot_only / proxy_behavioral -- pipeline validation only')


EXP-A Summary
  accuracy : 1.0000 +/- 0.0000
  f1       : 1.0 +/- 0.0
  roc_auc  : 1.0 +/- 0.0
  epochs   : 200.0 +/- 0.0


In [10]:
exp_b_results=[]; exp_b_summary=None

if TAG_AVAILABLE:
    print('EXP-B: LSTM Sequence + TAG Features')
    for seed in SEEDS:
        print(f'  Seed {seed:3d} ...', end=' ', flush=True)
        r = run_seed(seed, 'seq_tag',
                     X_train_full, y_train_full, mask_train_full, train_learner_groups,
                     X_test, y_test, mask_test,
                     tag_pool=tag_train_full, tag_te=tag_test)
        m = r['metrics']
        auc = f"{m['roc_auc']:.4f}" if m['roc_auc'] is not None else 'NA'
        print(f"acc={m['accuracy']:.4f}  f1={m.get('f1')}  auc={auc}  epochs={m['epochs_trained']}")
        exp_b_results.append(r)
    exp_b_summary = summarise_seeds(exp_b_results, 'EXP-B: LSTM Sequence + TAG')
    print(f"\nEXP-B acc: {exp_b_summary['accuracy_mean']:.4f} +/- {exp_b_summary['accuracy_std']:.4f}")
    print('WARNING: EXP-A vs EXP-B is descriptive only at pilot scale')
else:
    print('EXP-B skipped -- TAG artifacts not available')


EXP-B: LSTM Sequence + TAG Features
  Seed  11 ... 

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

acc=0.5000  f1=0.6666666666666666  auc=0.0741  epochs=16
  Seed  22 ... 

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

...

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

acc=1.0000  f1=1.0  auc=1.0000  epochs=200

EXP-B acc: 0.9000 +/- 0.2000


C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

In [11]:
s42_first = next((r for r in exp_a_results if r['seed']==42), None)
if s42_first:
    r2 = run_seed(42, 'seq_only',
                  X_train_full, y_train_full, mask_train_full, train_learner_groups,
                  X_test, y_test, mask_test)
    orig  = np.array(s42_first['y_pred_prob'])
    repro = np.array(r2['y_pred_prob'])
    max_diff = float(np.abs(orig - repro).max())
    repro_ok = max_diff < 1e-4
    print(f'Reproducibility (seed=42): max_pred_diff={max_diff:.2e} -> {"PASS" if repro_ok else "SOFT WARN"}')
    if not repro_ok:
        print('  PyTorch CPU non-determinism -- treating as soft warning')
else:
    repro_ok=True; max_diff=0.0; print('Seed 42 not in SEEDS -- skipped')


C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

...

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

Reproducibility (seed=42): max_pred_diff=0.00e+00 -> PASS


C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum(dim=1).clamp(min=1), dtype=torch.int64)
C:\Users\n.sukkhadamrongrak\AppData\Local\Temp\ipykernel_15284\2019638007.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lens  = torch.tensor(mb.sum

In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

lr = exp_a_results[-1]
h  = lr['history']
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(h['loss'],label='train'); axes[0].plot(h['val_loss'],label='val',ls='--')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title(f'EXP-A Loss (seed={lr["seed"]})'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(h['accuracy'],label='train'); axes[1].plot(h['val_accuracy'],label='val',ls='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'EXP-A Accuracy (seed={lr["seed"]})'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('PILOT -- proxy_behavioral / pilot_only labels', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig(REP_DIR / 'lstm_training_curves.png', dpi=150); plt.close()
print(f'Saved: lstm_training_curves.png')


Saved: lstm_training_curves.png


In [13]:
from sklearn.metrics import ConfusionMatrixDisplay
s42 = next((r for r in exp_a_results if r['seed']==42), exp_a_results[0])
y_pred_42 = (np.array(s42['y_pred_prob']) >= 0.5).astype(int)
cm_42 = confusion_matrix(s42['y_true'], y_pred_42)
fig,ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(cm_42, display_labels=['at_risk=0','at_risk=1']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'EXP-A Confusion Matrix (seed={s42["seed"]}) [PILOT]', fontsize=9)
plt.tight_layout()
plt.savefig(REP_DIR / 'lstm_confusion_matrix.png', dpi=150); plt.close()
print(f'Confusion matrix: {cm_42.tolist()}')


Confusion matrix: [[9, 0], [0, 9]]


In [14]:
from sklearn.metrics import roc_curve
y_true_42 = np.array(s42['y_true']); y_prob_42 = np.array(s42['y_pred_prob'])
ncls_42 = len(np.unique(y_true_42))
if ncls_42 >= 2:
    fpr,tpr,_ = roc_curve(y_true_42, y_prob_42)
    auc_42 = s42['metrics']['roc_auc']
    fig,ax = plt.subplots(figsize=(5,4))
    ax.plot(fpr,tpr, label=f'AUC={auc_42:.4f}' if auc_42 else 'AUC=NA')
    ax.plot([0,1],[0,1],'k--',alpha=0.4)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'EXP-A ROC (seed={s42["seed"]}) [PILOT]', fontsize=9)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(REP_DIR / 'lstm_roc_curve.png', dpi=150); plt.close()
    print(f'ROC saved.  AUC={auc_42}')
else:
    print(f'ROC NOT generated: test has {ncls_42} class(es) -- ROC-AUC is NA (undefined)')


ROC saved.  AUC=1.0


In [15]:
fig,ax = plt.subplots(figsize=(10,3))
for r in exp_a_results:
    ax.scatter(range(len(r['y_pred_prob'])), r['y_pred_prob'],
               alpha=0.5, s=20, label=f'seed={r["seed"]}')
ax.axhline(0.5, color='red', ls='--', lw=0.8, label='threshold=0.5')
ax.set_xlabel('Test seq index'); ax.set_ylabel('P(at_risk=1)')
ax.set_title('EXP-A Predicted probabilities across seeds [PILOT]', fontsize=9)
ax.legend(fontsize=7); ax.set_ylim([-0.05,1.05])
plt.tight_layout()
plt.savefig(REP_DIR / 'lstm_pred_probs.png', dpi=150); plt.close()
print(f'Saved: lstm_pred_probs.png')


Saved: lstm_pred_probs.png


In [16]:
config = {
    'schema_version':SCHEMA_VERSION, 'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'phase3_source_sha':PHASE3_SOURCE_SHA, 'seeds':SEEDS,
    'val_split_size':VAL_SPLIT_SIZE, 'max_epochs':MAX_EPOCHS,
    'early_stop_patience':EARLY_STOP_PATIENCE, 'lstm_units':LSTM_UNITS,
    'dropout_rate':DROPOUT_RATE, 'learning_rate':LEARNING_RATE, 'batch_size':BATCH_SIZE,
    'backend':'pytorch', 'architecture':'pack_padded -> LSTM -> Dropout -> Linear(sigmoid)',
    'n_features':N_FEATURES, 'max_seq_len':MAX_LEN, 'n_tag_features':n_tag,
    'label_source':'proxy_behavioral', 'label_validity':'pilot_only',
    'data_warning':'PILOT ONLY -- 10 learners, proxy labels. Not thesis results.',
}
config_path = MODEL_DIR / 'lstm_config_v1.json'
config_path.write_text(json.dumps(config, indent=2))
print(f'Config: {config_path}')

all_hist = [{'seed':r['seed'],'experiment':r['experiment'],'history':r['history']}
             for r in exp_a_results + exp_b_results]
hist_path = MODEL_DIR / 'lstm_history_v1.json'
hist_path.write_text(json.dumps(all_hist, indent=2))
print(f'History: {hist_path}')

pred_rows = [
    {'experiment':r['experiment'],'seed':r['seed'],'sequence_id':sid,
     'y_pred_prob':float(prob),'y_pred':int(prob>=0.5),'y_true':int(tl),
     'label_source':'proxy_behavioral','label_validity':'pilot_only'}
    for r in exp_a_results + exp_b_results
    for sid,prob,tl in zip(test_seq_ids, r['y_pred_prob'], r['y_true'])
]
pred_df = pd.DataFrame(pred_rows)
pred_path = MODEL_DIR / 'lstm_predictions_v1.parquet'
pred_df.to_parquet(pred_path, index=False)
print(f'Predictions: {pred_path}  ({len(pred_df)} rows)')

metrics_out = {'schema_version':SCHEMA_VERSION,
               'label_source':'proxy_behavioral','label_validity':'pilot_only',
               'data_warning':'PILOT ONLY -- Not thesis results.',
               'experiments':{'EXP-A':exp_a_summary}}
if exp_b_summary:
    metrics_out['experiments']['EXP-B'] = exp_b_summary
metrics_path = MODEL_DIR / 'lstm_metrics_v1.json'
metrics_path.write_text(json.dumps(metrics_out, indent=2, default=str))
print(f'Metrics: {metrics_path}')

print('\nEXP-A per-seed table:')
rows=[{'seed':r['metrics']['seed'],'acc':r['metrics']['accuracy'],
       'prec':r['metrics']['precision'],'rec':r['metrics']['recall'],
       'f1':r['metrics']['f1'],'auc':r['metrics']['roc_auc'],
       'epochs':r['metrics']['epochs_trained'],'train_s':round(r['metrics']['train_time_sec'],2)}
      for r in exp_a_results]
display(pd.DataFrame(rows))


Config: models\sequence\lstm\lstm_config_v1.json
History: models\sequence\lstm\lstm_history_v1.json
Predictions: models\sequence\lstm\lstm_predictions_v1.parquet  (180 rows)
Metrics: models\sequence\lstm\lstm_metrics_v1.json

EXP-A per-seed table:


,seed,acc,prec,rec,f1,auc,epochs,train_s
0,11,1.0,1.0,1.0,1.0,1.0,200,2.33
1,22,1.0,1.0,1.0,1.0,1.0,200,2.43
2,33,1.0,1.0,1.0,1.0,1.0,200,2.85
3,42,1.0,1.0,1.0,1.0,1.0,200,3.24
4,55,1.0,1.0,1.0,1.0,1.0,200,3.03


In [17]:
all_checks = list(pre_checks)
def chk(name, passed, detail=''):
    all_checks.append({'check':name,'result':'PASS' if passed else 'FAIL','detail':str(detail)})

chk('Test set not used for early stopping', True,
    'EarlyStopping monitors val_loss from val learners only')

all_preds = [p for r in exp_a_results for p in r['y_pred_prob']]
chk('Predictions within [0,1]',
    all(0.0<=p<=1.0 for p in all_preds),
    f'min={min(all_preds):.4f}  max={max(all_preds):.4f}')

n_test = X_test.shape[0]
chk('Prediction count == test sequences',
    all(len(r['y_pred_prob'])==n_test for r in exp_a_results),
    f'expected={n_test}')

chk('Timing values non-negative',
    all(r['metrics']['train_time_sec']>=0 and r['metrics']['inf_time_per_seq_sec']>=0
        for r in exp_a_results))

chk('All metrics labeled pilot_only',
    all(r['metrics'].get('label_validity')=='pilot_only' for r in exp_a_results))

chk('Re-run same seed is reproducible', repro_ok, f'max_pred_diff={max_diff:.2e}')

result_df   = pd.DataFrame(all_checks)
n_hard_fail = sum(1 for c in all_checks if c['result']=='FAIL')

print('\n-- M4 Validation Summary --')
print(result_df.to_string(index=False))
print(f'\n{len(all_checks)-n_hard_fail}/{len(all_checks)} checks passed')

if n_hard_fail > 0:
    raise RuntimeError(f'M4 validation FAILED -- {n_hard_fail} check(s).')

print('\nM4 COMPLETE -- Pilot LSTM pipeline validated.')
print('WARNING PILOT LIMITATIONS:')
print(f'  label_source=proxy_behavioral / label_validity=pilot_only')
print(f'  Learners={len(split_ledger)} total ({len(train_learners)} train, {len(test_learners)} test)')
print(f'  Tensors: train={X_train_full.shape}  test={X_test.shape}')
print(f'  These results are NOT final Chapter 4 conclusions.')
print(f'  Thesis requires >=60 learners with teacher-reviewed labels.')



-- M4 Validation Summary --
                               check result                                                 detail
      Input checksums in M2 manifest   PASS                                              6 entries
            Tensor shape valid (3-D)   PASS                     train=(72, 6, 10) test=(18, 6, 10)
   Padding mask shape matches tensor   PASS                                     train_mask=(72, 6)
               No NaN/Inf in tensors   PASS                                                       
              Frozen split unchanged   PASS                                       train=72 test=18
                  No learner overlap   PASS                                              overlap=0
 Scaler fitted on training data only   PASS                                        fit_split=train
          No blacklisted fields in X   PASS                                                  clean
Test set not used for early stopping   PASS EarlyStopping monitors val_loss from

In [18]:
def sha256_file(p):
    h = hashlib.sha256(); h.update(p.read_bytes()); return h.hexdigest()[:16]

artifact_files = [config_path, hist_path, pred_path, metrics_path]
lstm_manifest = {
    'schema_version':SCHEMA_VERSION,
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'phase3_source_sha':PHASE3_SOURCE_SHA,
    'm2_manifest_sha':sha256_file(manifest_path),
    'backend':'pytorch',
    'experiments_run':['EXP-A'] + (['EXP-B'] if exp_b_results else []),
    'seeds':SEEDS,
    'dataset_stats':{
        'train_learners':len(train_learners),'test_learners':len(test_learners),
        'train_sequences':int(X_train_full.shape[0]),
        'test_sequences':int(X_test.shape[0]),
        'tensor_shape_train':list(X_train_full.shape),
        'tensor_shape_test':list(X_test.shape),
        'n_features':N_FEATURES,'max_seq_len':MAX_LEN,
        'train_class_dist':{str(k):int(v) for k,v in zip(*np.unique(y_train_full,return_counts=True))},
        'test_class_dist':{str(k):int(v) for k,v in zip(*np.unique(y_test,return_counts=True))},
        'label_source':'proxy_behavioral','label_validity':'pilot_only',
    },
    'exp_a_summary':{k:v for k,v in exp_a_summary.items() if k!='per_seed'},
    'exp_b_summary':({k:v for k,v in exp_b_summary.items() if k!='per_seed'} if exp_b_summary else None),
    'validation_checks':len(all_checks),
    'validation_passed':len(all_checks)-n_hard_fail,
    'reproducibility':{'max_pred_diff':max_diff,'pass':repro_ok},
    'artifacts':{f.name:str(f) for f in artifact_files},
    'artifact_checksums':{f.name:sha256_file(f) for f in artifact_files},
    'data_warning':('PILOT ONLY -- 10 learners, proxy_behavioral. '
                    'Thesis requires teacher-reviewed labels and >=60 participants.'),
}
manifest_out = MODEL_DIR / 'lstm_manifest_v1.json'
manifest_out.write_text(json.dumps(lstm_manifest, indent=2, default=str))
print(f'LSTM manifest: {manifest_out}')

print('\n-- Artifact Summary --')
for f in artifact_files + [manifest_out]:
    print(f'  {f.name:<40s}  {f.stat().st_size/1024:7.1f} KB')
print('\nReport plots:')
for f in sorted(REP_DIR.glob('*.png')): print(f'  {f.name}')


LSTM manifest: models\sequence\lstm\lstm_manifest_v1.json

-- Artifact Summary --
  lstm_config_v1.json                           0.7 KB
  lstm_history_v1.json                        160.7 KB
  lstm_predictions_v1.parquet                   6.3 KB
  lstm_metrics_v1.json                          9.9 KB
  lstm_manifest_v1.json                         2.9 KB

Report plots:
  lstm_confusion_matrix.png
  lstm_pred_probs.png
  lstm_roc_curve.png
  lstm_training_curves.png
